<a href="https://colab.research.google.com/github/todd-jang/AIFFEL_quest_rs/blob/main/MainQuest/Quest04/AgenticAI/B2B_AutoGen_AgenticAI_deepseek.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Track B — AutoGen Agentic AI 실습

**주제**: 멀티에이전트 협업 + LangGraph 협업 브릿지

**학습 목표**
- AutoGen의 대화 기반 워크플로우 이해
- GroupChat으로 역할 기반 협업 구현
- 번역 + 자기검토 루프를 에이전트 팀으로
- Track A(LangGraph) 초안을 이어받아 최종 PDF 완성 (L2 소통)

**환경**: Antigravity IDE + Colab Extension (Colab 웹 호환)

---

## 실습 순서 (Track A와의 협업)

이 노트북은 Track A(LangGraph)와 파일로 연결됩니다.

1. Track A 노트북 실행 → `workspace/reports/trackA_draft.md` 생성
2. Track B 노트북 실행 → 초안을 받아 최종 PDF 완성

**Track A 노트북 마지막에 추가할 셀** (Track A에 아직 없다면 붙여넣기):

```python
from pathlib import Path
WORKDIR = Path.cwd() / "workspace"
draft_result = report_graph.invoke(
    {"messages": [("user", "모두의연구소에 대해 조사해서 리포트 초안을 작성해줘")]},
    config=RunnableConfig(recursion_limit=15, configurable={"thread_id": "bridge-1"})
)
trackA_final_text = draft_result["messages"][-1].content
draft_path = WORKDIR / "reports" / "trackA_draft.md"
draft_path.write_text(trackA_final_text, encoding="utf-8")
print(f"Track A 초안 저장: {draft_path}")
```

## 목차
- [0] 환경 설정
- [1] AutoGen 개념 + LangGraph 비교
- [2] 2인 대화 (Coder + Executor)
- [3] GroupChat 버그수정 워크플로우
- [4] 번역 + 자기검토 GroupChat
- [5] Track A → Track B 브릿지 (핵심)
- [6] L1/L2/L3 소통 정리

## 0. 환경 설정

In [1]:
import subprocess, sys
def pip_install(*packages):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)

print("Python:", sys.version.split()[0])

# 'autogen' 모듈을 제공하는 건 pyautogen 0.2.x 또는 ag2(0.x) 입니다.
# - pyautogen 0.2.x는 Python 3.12 이하만 지원, 최신 pyautogen(0.10+)은 모듈명이 달라 import autogen 불가
# - ag2 0.14 는 기존 AutoGen 0.2 API 그대로 + Python 3.10~3.13 지원 → ag2 사용
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "pyautogen"], capture_output=True)
pip_install(
    "ag2[openai]>=0.9,<1",
    "fpdf2",
    "pdfplumber",
    "requests",
)
print("설치 완료")

Python: 3.13.15
설치 완료


In [2]:
import os, getpass
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY: ")
print("키 설정 완료")

OPENAI_API_KEY: ··········
키 설정 완료


In [3]:
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path

WORKDIR = Path.cwd() / "workspace"
for sub in ("fonts", "reports", "coding"):
    (WORKDIR / sub).mkdir(parents=True, exist_ok=True)
print("WORKDIR:", WORKDIR.resolve())

WORKDIR: /content/workspace


In [4]:
import sys, subprocess
try:
    import autogen
except ModuleNotFoundError:
    # 설치 셀을 건너뛰었거나 런타임이 초기화된 경우 → 이 커널에 바로 설치 후 재시도
    print("autogen 없음 → 설치합니다 (1~2분 소요)")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "ag2[openai]>=0.9,<1"], check=True)
    import autogen
from autogen import AssistantAgent, UserProxyAgent, GroupChat, GroupChatManager

config_list = [{
    "model": "gpt-4o-mini",
    "api_key": os.environ["OPENAI_API_KEY"],
}]
llm_config = {
    "config_list": config_list,
    "temperature": 0.0,
    "cache_seed": None,
}

def has_word(msg, word):
    """content가 None이어도 안전하게 검사"""
    return word in (msg.get("content") or "")

# GroupChat 시작용 Admin: 그룹 멤버가 아니므로 첫 번째 멤버부터 발언이 시작됨
admin = UserProxyAgent(
    name="Admin",
    human_input_mode="NEVER",
    code_execution_config=False,
    max_consecutive_auto_reply=0,
)
print("AutoGen:", autogen.__version__)

AutoGen: 0.14.0


In [5]:
import requests
from fpdf import FPDF

# 가변폰트(NotoSansKR[wght])는 fpdf2에서 불안정 → 정적 TTF(NanumGothic) 사용
FONT_PATH = WORKDIR / "fonts" / "NanumGothic-Regular.ttf"
if not FONT_PATH.exists():
    url = "https://github.com/google/fonts/raw/main/ofl/nanumgothic/NanumGothic-Regular.ttf"
    resp = requests.get(url, timeout=60)
    resp.raise_for_status()  # 실패 시 HTML이 폰트 파일로 저장되는 것 방지
    FONT_PATH.write_bytes(resp.content)
    print("폰트 다운로드 완료")

def save_pdf(content: str, filename: str = "report.pdf") -> str:
    pdf = FPDF()
    pdf.add_page()
    pdf.set_auto_page_break(auto=True, margin=15)
    pdf.add_font("NanumGothic", "", str(FONT_PATH))  # uni=True는 fpdf2에서 deprecated
    pdf.set_font("NanumGothic", size=12)
    for line in content.split("\n"):
        if not line.strip():
            pdf.ln(4)
            continue
        # new_x="LMARGIN": 줄바꿈 후 왼쪽 여백으로 복귀 (없으면 두 번째 줄에서 예외 발생)
        pdf.multi_cell(0, 8, line, new_x="LMARGIN", new_y="NEXT")
    out_path = WORKDIR / "reports" / filename
    pdf.output(str(out_path))
    return str(out_path)

print("save_pdf 준비 완료")

폰트 다운로드 완료
save_pdf 준비 완료


## 1. AutoGen 개념 + LangGraph 비교

| 관점 | LangGraph (A) | AutoGen (B) |
|------|---------------|-------------|
| 흐름 제어 | 명시적 (엣지) | 암묵적 (대화) |
| 상태 관리 | State TypedDict | chat.messages |
| emergent | 제한적 | 강함 |
| 디버깅 | 노드별 추적 | 로그 분석 |
| 러닝커브 | 그래프 개념 | 직관적 |

## 2. 2인 대화 (Coder + Executor)

In [6]:
coder = AssistantAgent(
    name="Coder",
    llm_config=llm_config,
    system_message=(
        "당신은 파이썬 개발자입니다. "
        "코드는 ```python 블록으로 감싸세요. "
        "작업 완료 시 'TERMINATE'를 출력하세요."
    ),
)

executor = UserProxyAgent(
    name="Executor",
    human_input_mode="NEVER",
    code_execution_config={
        "work_dir": str(WORKDIR / "coding"),
        "use_docker": False,
    },
    max_consecutive_auto_reply=3,
    is_termination_msg=lambda x: (x.get("content") or "").rstrip().endswith("TERMINATE"),
)

executor.initiate_chat(
    coder,
    message="1부터 100까지 더하는 파이썬 코드를 작성하고 실행해서 결과를 알려줘.",
)

Executor (to Coder):

1부터 100까지 더하는 파이썬 코드를 작성하고 실행해서 결과를 알려줘.

--------------------------------------------------------------------------------
Coder (to Executor):

```python
# 1부터 100까지 더하는 코드
total = sum(range(1, 101))
total
```

위 코드를 실행하면 1부터 100까지의 합은 5050입니다. 

TERMINATE

--------------------------------------------------------------------------------

>>>>>>>> TERMINATING RUN (27860cae-4588-427b-bfb2-878ceba217a1): Termination message condition on agent 'Executor' met


ChatResult(chat_id=4055168300605283218079463545614039207, chat_history=[{'content': '1부터 100까지 더하는 파이썬 코드를 작성하고 실행해서 결과를 알려줘.', 'role': 'assistant', 'name': 'Executor'}, {'content': '```python\n# 1부터 100까지 더하는 코드\ntotal = sum(range(1, 101))\ntotal\n```\n\n위 코드를 실행하면 1부터 100까지의 합은 5050입니다. \n\nTERMINATE', 'role': 'user', 'name': 'Coder'}], summary='```python\n# 1부터 100까지 더하는 코드\ntotal = sum(range(1, 101))\ntotal\n```\n\n위 코드를 실행하면 1부터 100까지의 합은 5050입니다. \n\n', cost={'usage_including_cached_inference': {'total_cost': 4.095e-05, 'gpt-4o-mini-2024-07-18': {'cost': 4.095e-05, 'prompt_tokens': 69, 'completion_tokens': 51, 'total_tokens': 120}}, 'usage_excluding_cached_inference': {'total_cost': 4.095e-05, 'gpt-4o-mini-2024-07-18': {'cost': 4.095e-05, 'prompt_tokens': 69, 'completion_tokens': 51, 'total_tokens': 120}}}, human_input=[])

## 3. GroupChat 버그수정 워크플로우 (Triage / Patcher / Reviewer / Logger)

In [7]:
triage = AssistantAgent(
    name="Triage",
    llm_config=llm_config,
    system_message=(
        "당신은 버그 분류 담당자입니다. "
        "카테고리와 우선순위를 한 줄로 요약하고 'TRIAGE_DONE'으로 끝맺으세요."
    ),
)

patcher = AssistantAgent(
    name="Patcher",
    llm_config=llm_config,
    system_message=(
        "당신은 패치 개발자입니다. "
        "파이썬 패치 코드를 ```python 블록으로 작성하고 'PATCH_DONE'으로 끝맺으세요."
    ),
)

reviewer = AssistantAgent(
    name="Reviewer",
    llm_config=llm_config,
    system_message=(
        "당신은 코드 리뷰어입니다. "
        "8점 이상이면 'APPROVED', 미만이면 'REVISE'로만 답하세요."
    ),
)

# LLM이 없는 UserProxyAgent는 발언을 생성할 수 없음 → AssistantAgent로 변경
logger = AssistantAgent(
    name="Logger",
    llm_config=llm_config,
    system_message=(
        "당신은 기록 담당자입니다. 논의를 3줄로 요약하고 마지막에 'TERMINATE'를 출력하세요."
    ),
)

groupchat = GroupChat(
    agents=[triage, patcher, reviewer, logger],
    messages=[],
    max_round=5,  # Admin 첫 메시지 1 + Triage → Patcher → Reviewer → Logger 4
    speaker_selection_method="round_robin",
)
manager = GroupChatManager(
    groupchat=groupchat,
    llm_config=llm_config,
    is_termination_msg=lambda x: has_word(x, "TERMINATE"),
)

# Triage가 아닌 Admin이 시작해야 Triage가 첫 발언자가 됨
admin.initiate_chat(
    manager,
    message=("다음 버그를 수정하세요: '리스트에서 값을 찾을 때 "
             "len(list)-1 인덱스 접근 시 IndexError가 발생.'"),
)

Admin (to chat_manager):

다음 버그를 수정하세요: '리스트에서 값을 찾을 때 len(list)-1 인덱스 접근 시 IndexError가 발생.'

--------------------------------------------------------------------------------

Next speaker: Triage

Triage (to chat_manager):

카테고리: 인덱스 오류, 우선순위: 중간. TRIAGE_DONE

--------------------------------------------------------------------------------

Next speaker: Patcher

Patcher (to chat_manager):

```python
def safe_access_list(my_list, index):
    if index < 0 or index >= len(my_list):
        raise IndexError("Index out of bounds")
    return my_list[index]

# 예시 사용
my_list = [1, 2, 3, 4, 5]
try:
    value = safe_access_list(my_list, len(my_list) - 1)  # 마지막 요소 접근
    print(value)
except IndexError as e:
    print(e)

PATCH_DONE
```

--------------------------------------------------------------------------------

Next speaker: Reviewer

Reviewer (to chat_manager):

APPROVED

--------------------------------------------------------------------------------

Next speaker: Logger

Logger (to 

ChatResult(chat_id=122748393538012526100602070936696991306, chat_history=[{'content': "다음 버그를 수정하세요: '리스트에서 값을 찾을 때 len(list)-1 인덱스 접근 시 IndexError가 발생.'", 'role': 'assistant', 'name': 'Admin'}], summary="다음 버그를 수정하세요: '리스트에서 값을 찾을 때 len(list)-1 인덱스 접근 시 IndexError가 발생.'", cost={'usage_including_cached_inference': {'total_cost': 0}, 'usage_excluding_cached_inference': {'total_cost': 0}}, human_input=[])

## 4. 번역 + 자기검토 GroupChat (Translator / Reviewer / Retranslator)

In [8]:
SAMPLE_EN = (
    "Hyundai Motor Company announced its Q4 2023 business results. "
    "Annual revenue increased 14.4% to KRW 162.7 trillion. "
    "Operating profit rose 54% to KRW 15.12 trillion."
)

translator = AssistantAgent(
    name="Translator",
    llm_config=llm_config,
    system_message=(
        "당신은 다국어 번역가입니다. 반드시 한국어로 번역하세요. "
        "고유명사는 '한국어(원어)' 형식. "
        "숫자/통화는 원문 + 한국식 병기. "
        "번역만 출력하고 'TRANSLATE_DONE'으로 끝맺으세요."
    ),
)

reviewer_tr = AssistantAgent(
    name="Reviewer",
    llm_config=llm_config,
    system_message=(
        "당신은 번역 검토자입니다. "
        "완전성(3)/정확성(3)/자연스러움(2)/일관성(2) 총 10점. "
        "8점 이상이면 'APPROVED', 미만이면 'REVISE'와 이유를 출력하세요."
    ),
)

retranslator = AssistantAgent(
    name="Retranslator",
    llm_config=llm_config,
    system_message=(
        "당신은 재번역가입니다. 지적된 문제를 반영해 재번역하고 'RETRANSLATE_DONE'으로 끝맺으세요."
    ),
)

groupchat_tr = GroupChat(
    agents=[translator, reviewer_tr, retranslator],
    messages=[],
    max_round=4,  # Admin 첫 메시지 1 + Translator → Reviewer → Retranslator 3
    speaker_selection_method="round_robin",
)
manager_tr = GroupChatManager(groupchat=groupchat_tr, llm_config=llm_config)

# Admin이 시작해야 Translator가 실제로 번역부터 수행함
admin.initiate_chat(
    manager_tr,
    message=f"다음 텍스트를 한국어로 번역하고 검토하세요:\n\n{SAMPLE_EN}",
)

Admin (to chat_manager):

다음 텍스트를 한국어로 번역하고 검토하세요:

Hyundai Motor Company announced its Q4 2023 business results. Annual revenue increased 14.4% to KRW 162.7 trillion. Operating profit rose 54% to KRW 15.12 trillion.

--------------------------------------------------------------------------------

Next speaker: Translator

Translator (to chat_manager):

현대자동차가 2023년 4분기 사업 결과를 발표했습니다. 연간 매출은 14.4% 증가하여 162.7조 원(KRW 162.7 trillion)에 달했습니다. 운영 이익은 54% 상승하여 15.12조 원(KRW 15.12 trillion)에 이르렀습니다. 

TRANSLATE_DONE

--------------------------------------------------------------------------------

Next speaker: Reviewer

Reviewer (to chat_manager):

검토 결과:

완전성: 3  
정확성: 3  
자연스러움: 2  
일관성: 2  

총점: 10점

결과: APPROVED

이유: 번역이 전체적으로 정확하고, 필요한 모든 정보를 포함하고 있습니다. 그러나 '운영 이익'이라는 표현이 '영업 이익'으로 더 자연스럽게 표현될 수 있으며, 'KRW'를 한국어로 '원'으로 표기하는 것이 더 일관적일 수 있습니다. 그럼에도 불구하고 전반적으로 이해하기 쉬운 번역입니다.

--------------------------------------------------------------------------------

Next speaker: Retranslator

Retran

ChatResult(chat_id=131460598747634718851524432669253719974, chat_history=[{'content': '다음 텍스트를 한국어로 번역하고 검토하세요:\n\nHyundai Motor Company announced its Q4 2023 business results. Annual revenue increased 14.4% to KRW 162.7 trillion. Operating profit rose 54% to KRW 15.12 trillion.', 'role': 'assistant', 'name': 'Admin'}], summary='다음 텍스트를 한국어로 번역하고 검토하세요:\n\nHyundai Motor Company announced its Q4 2023 business results. Annual revenue increased 14.4% to KRW 162.7 trillion. Operating profit rose 54% to KRW 15.12 trillion.', cost={'usage_including_cached_inference': {'total_cost': 0}, 'usage_excluding_cached_inference': {'total_cost': 0}}, human_input=[])

## 5. Track A → Track B 브릿지 (L2 소통)

Track A(LangGraph)의 초안을 AutoGen Editor 팀이 이어받아 최종 PDF로 저장합니다.
Track A 초안 파일이 없으면 샘플 초안으로 진행합니다.

In [9]:
draft_path = WORKDIR / "reports" / "trackA_draft.md"

if draft_path.exists():
    langgraph_draft = draft_path.read_text(encoding="utf-8")
    print(f"Track A 초안 로드: {len(langgraph_draft)}자")
else:
    langgraph_draft = (
        "# 모두의연구소 리포트 (샘플 초안)\n"
        "## 1. 개요\n"
        "모두의연구소는 2015년 설립된 커뮤니티 기반 AI 교육 스타트업입니다.\n"
        "## 2. 주요 사업\n"
        "- AI 교육 (아이펠)\n"
        "- 연구 커뮤니티 (페이퍼샵)\n"
        "## 3. 인사이트\n"
        "- 커뮤니티 기반 학습 모델의 차별성\n"
        "- 70억 원 투자 유치\n"
    )
    print("Track A 초안 없음 → 샘플 초안 사용")

print("--- 초안 앞 300자 ---")
print(langgraph_draft[:300])

Track A 초안 없음 → 샘플 초안 사용
--- 초안 앞 300자 ---
# 모두의연구소 리포트 (샘플 초안)
## 1. 개요
모두의연구소는 2015년 설립된 커뮤니티 기반 AI 교육 스타트업입니다.
## 2. 주요 사업
- AI 교육 (아이펠)
- 연구 커뮤니티 (페이퍼샵)
## 3. 인사이트
- 커뮤니티 기반 학습 모델의 차별성
- 70억 원 투자 유치



In [10]:
researcher = AssistantAgent(
    name="Researcher",
    llm_config=llm_config,
    system_message=(
        "당신은 리서치 검증자입니다. 사실 확인이 필요한 항목을 최대 3개 지적하고 'RESEARCH_DONE'으로 끝맺으세요."
    ),
)

editor = AssistantAgent(
    name="Editor",
    llm_config=llm_config,
    system_message=(
        "당신은 한국어 리포트 편집자입니다. 지적을 반영해 리포트 전문을 마크다운으로 재작성하세요. "
        "코드블록(```)으로 감싸지 말고, 마지막 줄에 'EDITOR_DONE'을 붙이세요."
    ),
)

fact_checker = AssistantAgent(
    name="FactChecker",
    llm_config=llm_config,
    system_message=(
        "당신은 팩트체커입니다. 숫자/날짜/고유명사 오류를 확인하세요. "
        "8점 이상이면 'APPROVED', 미만이면 'REVISE'라고만 답하세요."
    ),
)

# PDF 저장은 코드 실행 에이전트 대신 save_pdf()로 직접 수행 (아래 셀)
print("Editor 팀 구성 완료")

Editor 팀 구성 완료


In [11]:
groupchat_bridge = GroupChat(
    agents=[researcher, editor, fact_checker],
    messages=[],
    max_round=4,  # Admin 첫 메시지 1 + Researcher → Editor → FactChecker 3
    speaker_selection_method="round_robin",
)
manager_bridge = GroupChatManager(groupchat=groupchat_bridge, llm_config=llm_config)

bridge_message = (
    "다음은 Track A(LangGraph)가 작성한 초안입니다. "
    "검토 → 개선 → 검증 순서로 진행하세요.\n\n"
    f"[Track A 초안]\n{langgraph_draft}\n\n"
    "[지시]\n"
    "1. Researcher: 사실 확인 항목 지적\n"
    "2. Editor: 지적 반영해 리포트 전문 재작성\n"
    "3. FactChecker: 8점 이상이면 APPROVED\n"
)

# Admin이 시작해야 Researcher가 첫 발언자가 됨
admin.initiate_chat(manager_bridge, message=bridge_message)

Admin (to chat_manager):

다음은 Track A(LangGraph)가 작성한 초안입니다. 검토 → 개선 → 검증 순서로 진행하세요.

[Track A 초안]
# 모두의연구소 리포트 (샘플 초안)
## 1. 개요
모두의연구소는 2015년 설립된 커뮤니티 기반 AI 교육 스타트업입니다.
## 2. 주요 사업
- AI 교육 (아이펠)
- 연구 커뮤니티 (페이퍼샵)
## 3. 인사이트
- 커뮤니티 기반 학습 모델의 차별성
- 70억 원 투자 유치


[지시]
1. Researcher: 사실 확인 항목 지적
2. Editor: 지적 반영해 리포트 전문 재작성
3. FactChecker: 8점 이상이면 APPROVED


--------------------------------------------------------------------------------

Next speaker: Researcher

Researcher (to chat_manager):

1. 사실 확인 항목 지적:
   - "모두의연구소는 2015년 설립된 커뮤니티 기반 AI 교육 스타트업입니다." → 설립 연도 및 사업 모델의 정확성 확인 필요.
   - "AI 교육 (아이펠)" → 아이펠이 모두의연구소의 AI 교육 프로그램인지 확인 필요.
   - "70억 원 투자 유치" → 투자 유치 금액 및 시점 확인 필요.

RESEARCH_DONE

--------------------------------------------------------------------------------

Next speaker: Editor

Editor (to chat_manager):

# 모두의연구소 리포트 (샘플 초안)
## 1. 개요
모두의연구소는 2015년에 설립된 커뮤니티 기반 AI 교육 스타트업으로, AI 교육과 연구 커뮤니티를 중심으로 활동하고 있습니다.

## 2. 주요 사업
- AI 교육 프로그램인 아이펠
- 연구 커뮤니티인 페이퍼샵

## 3. 인사이트
- 커뮤니티 

ChatResult(chat_id=241650909198146959046838210016822047079, chat_history=[{'content': '다음은 Track A(LangGraph)가 작성한 초안입니다. 검토 → 개선 → 검증 순서로 진행하세요.\n\n[Track A 초안]\n# 모두의연구소 리포트 (샘플 초안)\n## 1. 개요\n모두의연구소는 2015년 설립된 커뮤니티 기반 AI 교육 스타트업입니다.\n## 2. 주요 사업\n- AI 교육 (아이펠)\n- 연구 커뮤니티 (페이퍼샵)\n## 3. 인사이트\n- 커뮤니티 기반 학습 모델의 차별성\n- 70억 원 투자 유치\n\n\n[지시]\n1. Researcher: 사실 확인 항목 지적\n2. Editor: 지적 반영해 리포트 전문 재작성\n3. FactChecker: 8점 이상이면 APPROVED\n', 'role': 'assistant', 'name': 'Admin'}], summary='다음은 Track A(LangGraph)가 작성한 초안입니다. 검토 → 개선 → 검증 순서로 진행하세요.\n\n[Track A 초안]\n# 모두의연구소 리포트 (샘플 초안)\n## 1. 개요\n모두의연구소는 2015년 설립된 커뮤니티 기반 AI 교육 스타트업입니다.\n## 2. 주요 사업\n- AI 교육 (아이펠)\n- 연구 커뮤니티 (페이퍼샵)\n## 3. 인사이트\n- 커뮤니티 기반 학습 모델의 차별성\n- 70억 원 투자 유치\n\n\n[지시]\n1. Researcher: 사실 확인 항목 지적\n2. Editor: 지적 반영해 리포트 전문 재작성\n3. FactChecker: 8점 이상이면 APPROVED\n', cost={'usage_including_cached_inference': {'total_cost': 0}, 'usage_excluding_cached_inference': {'total_cost': 0}}, human_input=[])

In [12]:
# Editor의 마지막 발언을 최종본으로 사용 → PDF 저장
def last_msg_of(name):
    msgs = [m for m in groupchat_bridge.messages if m.get("name") == name and m.get("content")]
    return msgs[-1]["content"] if msgs else None

editor_text = last_msg_of("Editor")
if editor_text:
    lines = [l for l in editor_text.splitlines() if not l.strip().startswith("```")]
    final_text = "\n".join(lines).replace("EDITOR_DONE", "").strip()
    print("FactChecker 판정:", (last_msg_of("FactChecker") or "없음").strip()[:30])
else:
    print("Editor 발언 없음 → Track A 초안 그대로 저장")
    final_text = langgraph_draft

final_pdf = Path(save_pdf(final_text, "bridge_final.pdf"))
print(f"최종 PDF: {final_pdf} ({final_pdf.stat().st_size:,} bytes)")

FactChecker 판정: REVISE
최종 PDF: /content/workspace/reports/bridge_final.pdf (21,604 bytes)


In [13]:
# 저장된 PDF 첫 페이지 텍스트 미리보기
import pdfplumber
with pdfplumber.open(final_pdf) as pdf:
    text = pdf.pages[0].extract_text() or ""
print("--- PDF 1페이지 앞 500자 ---")
print(text[:500])

--- PDF 1페이지 앞 500자 ---
# 모두의연구소 리포트 (샘플 초안)
## 1. 개요
모두의연구소는 2015년에 설립된 커뮤니티 기반 AI 교육 스타트업으로, AI 교육과 연구 커뮤니티를
중심으로 활동하고 있습니다.
## 2. 주요 사업
- AI 교육 프로그램인 아이펠
- 연구 커뮤니티인 페이퍼샵
## 3. 인사이트
- 커뮤니티 기반 학습 모델의 차별성
- 최근 70억 원 규모의 투자 유치


## 6. L1/L2/L3 소통 정리

### L1: 프레임워크 내부 소통 (emergent)
- Track A-4: State 공유 → 노드 순환
- Track B-2~4: GroupChat 메시지 큐 → 자동 순환

### L2: 프레임워크 간 파이프라인 소통 (오늘 실습)
- Track A(LangGraph) → `trackA_draft.md` → Track B(AutoGen)
- 파일/함수 반환값으로 배턴 터치

### L3: 실시간 양방향 소통 (개념)
- MCP / A2A / AGNTCY 프로토콜
- LangGraph: `langchain-mcp-adapters`
- AutoGen: `FastMCP`로 팀을 서버로 노출
- 별도 세션에서 다룰 것

### 강점 비교
| 강점 | LangGraph | AutoGen |
|------|:---:|:---:|
| 통제 | 강 | 약 |
| 유연성 | 약 | 강 |
| emergent | 제한 | 강 |
| 디버깅 | 쉬움 | 어려움 |

## 실습 완료

산출물: `workspace/reports/bridge_final.pdf`

다음 단계:
1. `speaker_selection_method="auto"`로 emergent 실험
2. MCP 브릿지 (L3) 별도 실습
3. Track A 자기검토 루프와 비교 분석